In [ ]:
# Script plots the change in the interannual standard deviation of extreme heat season length between periods in the CESM ensemble data. 
# Uses centered means (subtract mean from the yearly length values before calculating standard deviation...this is key given the permutation test).
# It can be used to re-create the change in standard deviation figures in the manuscript.

In [ ]:
import glob
import os
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import regionmask
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as mcolors


In [ ]:
# Indicate location where the CESM Heat Season Characteristics Files are located.
# Script is designed to work with one temperature variable (TMAX or TMIN) at a time.

OUTPUT_DIR = '/...TMMN/...'
#OUTPUT_DIR = '/...TMAX/...'

In [ ]:

def load_ensemble_data(period_tag):
    """
    Scans the output directory for processed files matching the period tag 
    ('Hist' or 'Fut'), loads them, and stacks them into a single Dataset.
    """
    search_pattern = os.path.join(OUTPUT_DIR, f"ExtremeHeat_*_{period_tag}*.nc")
    file_paths = sorted(glob.glob(search_pattern))
    
    if not file_paths:
        print(f"No files found for '{period_tag}'. Check your directory.")
        return None
        
    print(f"Found {len(file_paths)} files for the {period_tag} period. Stacking...")
    
    ds_list = []
    
    for file_path in file_paths:
        basename = os.path.basename(file_path)
        
        
        # Splitting 'ExtremeHeat_1011.001_Fut_2030_2059.nc' by '_' creates a list:
        # ['ExtremeHeat', '1011.001', 'Fut', '2030', '2059.nc']
        # We just grab the second item (index 1)
        member_id = basename.split('_')[1] 
        
        ds = xr.open_dataset(file_path)
        ds = ds.expand_dims(member=[member_id])
        ds_list.append(ds)
        
    ds_ensemble = xr.concat(ds_list, dim='member')
    print(f"Successfully created ensemble dataset with dimensions: {dict(ds_ensemble.dims)}")
    
    return ds_ensemble




print("--- Loading Historical Ensemble ---")
ens_hist_raw = load_ensemble_data("Hist_1966_1995")

print("\n--- Loading Future Ensemble ---")
ens_fut_raw = load_ensemble_data("Fut_2026_2055")

In [ ]:
# =============================================================================
# FDR
# =============================================================================
def apply_fdr(p_values, alpha=0.05):
    """
    Applies the Benjamini-Hochberg False Discovery Rate (FDR) procedure.
    """
    p_flat = p_values.values.flatten()
    valid_idx = ~np.isnan(p_flat)
    p_valid = p_flat[valid_idx]
    
    # Sort p-values
    sorted_idx = np.argsort(p_valid)
    p_sorted = p_valid[sorted_idx]
    
    # Calculate critical values
    m = len(p_sorted)
    q_values = (np.arange(1, m + 1) / m) * alpha
    
    # Find the largest p-value that is less than its critical value
    significant = p_sorted <= q_values
    if np.any(significant):
        max_idx = np.where(significant)[0][-1]
        p_threshold = p_sorted[max_idx]
    else:
        p_threshold = -1.0 # No significant pixels
        
    # Create the spatial mask
    return p_values <= p_threshold


In [ ]:
def calculate_std_emergence_robustness(da_hist, da_fut, agreement_threshold=0.666, n_permutations=1000):
    """
    Calculates significance of STANDARD DEVIATION on a member-by-member basis (shuffling years).
    A grid cell is robust if >= threshold (e.g., 66%) of members show a 
    statistically significant change in variance in the SAME direction as the ensemble mean.
    """
    print("1. Calculating Ensemble Standard Deviation Change & Direction...")
    # Calculate the standard deviation across years for each member
    mem_std_hist = da_hist.std(dim='year', skipna=True)
    mem_std_fut  = da_fut.std(dim='year', skipna=True)
    
    # Calculate difference in variability per member, then the overall ensemble mean
    mem_diff = mem_std_fut - mem_std_hist
    ens_mean_diff = mem_diff.mean(dim='member', skipna=True)
    
    # Get the overall direction of the ensemble mean change
    ens_sign = np.sign(ens_mean_diff.values)
    
    total_members = da_hist.sizes['member']
    n_years_hist = da_hist.sizes['year']
    n_years_total = n_years_hist + da_fut.sizes['year']
    
    # Initialize a blank map to count how many members pass the test
    robust_member_count = xr.zeros_like(ens_mean_diff)
    
    print(f"2. Running Member-by-Member Permutation Tests ({total_members} members, {n_permutations} shuffles each)...")
    print("   (Note: This may take a few minutes)")
    np.random.seed(42)
    
    for m in range(total_members):
        print(f"   Processing Member {m+1}/{total_members}...", end='\r')
        
        # Extract data for just this single member as pure numpy arrays for speed
        m_hist = da_hist.isel(member=m).values
        m_fut  = da_fut.isel(member=m).values
        m_obs_diff = mem_diff.isel(member=m).values
        
        # Get the direction of THIS member's change
        m_sign = np.sign(m_obs_diff)
        
        # MEAN-CENTERING BEFORE SHUFFLING
        # We subtract the temporal mean of the historical period from the historical data,
        # and the temporal mean of the future period from the future data.
        # This isolates the variance so we aren't shuffling the big shift in the mean.
        m_hist_centered = m_hist - np.nanmean(m_hist, axis=0)
        m_fut_centered  = m_fut - np.nanmean(m_fut, axis=0)
        
        # Combine the CENTERED years for shuffling (axis=0 is the year dimension)
        combined = np.concatenate([m_hist_centered, m_fut_centered], axis=0) 
        exceed_count = np.zeros_like(m_obs_diff)
        
        # Shuffle years for this specific member
        for _ in range(n_permutations):
            idx = np.random.permutation(n_years_total)
            shuffled = combined[idx, ...]
            
            pseudo_hist = np.nanstd(shuffled[:n_years_hist, ...], axis=0)
            pseudo_fut  = np.nanstd(shuffled[n_years_hist:, ...], axis=0)
            pseudo_diff = pseudo_fut - pseudo_hist
            
            exceed_count += (np.abs(pseudo_diff) >= np.abs(m_obs_diff))
            
        # Calculate p-values and apply FDR for THIS member only
        p_values = exceed_count / n_permutations
        p_val_da = xr.DataArray(p_values, coords=ens_mean_diff.coords, dims=ens_mean_diff.dims)
        m_fdr_mask = apply_fdr(p_val_da, alpha=0.05)
        
        # Check criteria: Is it significant AND does the sign match the ensemble average?
        sign_match = (m_sign == ens_sign)
        member_is_robust = m_fdr_mask & sign_match
        
        # Add the passing grid cells to our running tally
        robust_member_count += member_is_robust
        
    print(f"\n3. Applying {agreement_threshold*100:.1f}% Agreement Threshold...")
    final_emergence_mask = (robust_member_count / total_members) >= agreement_threshold
    
    return mem_diff, ens_mean_diff, final_emergence_mask

In [ ]:
da_len_hist = ens_hist_raw['season_length']
da_len_fut  = ens_fut_raw['season_length']

# Run the member-by-member STANDARD DEVIATION emergence script
mem_diff_std, ens_diff_std, final_mask_std = calculate_std_emergence_robustness(
    da_len_hist, 
    da_len_fut, 
    agreement_threshold=0.666, # 2/3 of members 
    n_permutations=1000
)

In [ ]:
def plot_robust_map_with_zonal(mem_diff, ens_diff, robust_mask, title, 
                               label='Shift in Days', cmap='PuOr_r', 
                               levels=None, xlim=None):
    
    print("1. Realigning Longitudes (0-360 to -180-180)...")
    def shift_lon(da):
        return da.assign_coords(lon=(((da.lon + 180) % 360) - 180)).sortby('lon')
        
    mem_diff = shift_lon(mem_diff)
    ens_diff = shift_lon(ens_diff)
    robust_mask = shift_lon(robust_mask)

    print("2. Applying Land Mask...")
    # Use ens_diff for the grid to generate the land mask
    land_mask = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(ens_diff)
    
    mem_diff_land = mem_diff.where(land_mask == 0)
    ens_diff_land = ens_diff.where(land_mask == 0)
    
    print("3. Calculating Zonal Means and Full Ensemble Spread (Land Only)...")
    zonal_mem_diff = mem_diff_land.mean(dim='lon', skipna=True)
    
    zonal_ens_mean = zonal_mem_diff.mean(dim='member', skipna=True)
    zonal_ens_min = zonal_mem_diff.min(dim='member', skipna=True)
    zonal_ens_max = zonal_mem_diff.max(dim='member', skipna=True)
    lats = zonal_ens_mean.lat

    print("4. Generating Aligned Figure...")
    fig, ax_map = plt.subplots(figsize=(12, 7), subplot_kw={'projection': ccrs.PlateCarree()})
    ax_map.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())
    
    divider = make_axes_locatable(ax_map)
    ax_zonal = divider.append_axes("right", size="20%", pad=0.3, axes_class=plt.Axes)
    cax = divider.append_axes("bottom", size="5%", pad=0.4, axes_class=plt.Axes)

    # ==========================================
    # LEFT: MASKED MAP PLOT
    # ==========================================
    robust_diff = ens_diff_land.where(robust_mask)

    map_plot = robust_diff.plot(
        ax=ax_map, 
        transform=ccrs.PlateCarree(),
        cmap=cmap, 
        levels=levels, 
        extend='both',
        add_colorbar=False,
        zorder=1
    )

    ax_map.add_feature(cfeature.OCEAN, facecolor='white', edgecolor='none', zorder=2)
    ax_map.coastlines(color='black', linewidth=0.8, zorder=3)
    ax_map.add_feature(cfeature.COASTLINE, linestyle=':', edgecolor='gray', linewidth=0.5, zorder=3)
    
    gl = ax_map.gridlines(draw_labels=True, xlocs=np.arange(-180, 181, 60), ylocs=np.arange(-90, 91, 30), color='lightgray', linewidth=0.8, linestyle='-', zorder=4)
    gl.top_labels = False
    gl.right_labels = False
    
    ax_map.set_title(title, fontsize=14, fontweight='bold', pad=15)
    
    cbar = plt.colorbar(map_plot, cax=cax, orientation='horizontal')
    cbar.set_label(label, fontsize=12)
    
    # ==========================================
    # RIGHT: ZONAL MEAN PLOT
    # ==========================================
    ax_zonal.axvline(0, color='black', linestyle='-', linewidth=1, zorder=2)
    ax_zonal.fill_betweenx(lats, zonal_ens_min, zonal_ens_max, color='gray', alpha=0.3, zorder=1)
    ax_zonal.plot(zonal_ens_mean, lats, color='red', linewidth=2, zorder=3)
    
    ax_zonal.set_ylim(-90, 90) 
    ax_zonal.set_xlim(xlim[0], xlim[1])
    ax_zonal.set_yticks(np.arange(-90, 100, 30))
    ax_zonal.set_yticklabels(['90°S', '60°S', '30°S', 'EQ', '30°N', '60°N', '90°N'])
    ax_zonal.yaxis.tick_right()
    
    ax_zonal.set_xlabel(label)
    ax_zonal.set_title('Land-Only Zonal Mean', fontsize=12, fontweight='bold')
    ax_zonal.grid(True, linestyle=':', alpha=0.6)
    
    plt.show()
    return fig

In [ ]:
# Plot the result
fig_std = plot_robust_map_with_zonal(
    mem_diff=mem_diff_std,     
    ens_diff=ens_diff_std,     
    robust_mask=final_mask_std,
    #title="Avg CESM Change in Global TMAX Season Length Standard Deviation (2026-2055) - (1966-1995)",
    title="Avg CESM Change in Global TMIN Season Length Standard Deviation (2026-2055) - (1966-1995)",
    label="Change in Standard Deviation (Days)",
    cmap="PuOr_r",             
    levels=np.arange(-60, 70, 10), 
    xlim=(-80, 80)             
)

#fig_std.savefig("plots/Centered_CESM_TMAX_Future_26-55_SeasonLength_Variability_Change.pdf", format="pdf", bbox_inches="tight")
fig_std.savefig("plots/Centered_CESM_TMIN_Future_26-55_SeasonLength_Variability_Change.pdf", format="pdf", bbox_inches="tight")